<a href="https://colab.research.google.com/github/sajina464/C-Programs/blob/main/day_13.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:

!pip -q install scikit-learn gradio transformers sentencepiece


In [2]:

import pandas as pd
from sklearn.ensemble import RandomForestClassifier

# Sample educational dataset
data = pd.DataFrame({
    "credit_score": [
        780, 750, 720, 700, 680, 650, 620, 600,
        580, 550, 740, 690, 630, 570, 710, 660,
        790, 610, 670, 540, 730, 640, 590, 560
    ],
    "income": [
        80000, 70000, 60000, 55000, 50000, 45000,
        40000, 35000, 30000, 25000, 65000, 48000,
        42000, 28000, 58000, 44000, 90000, 36000,
        47000, 22000, 62000, 39000, 32000, 26000
    ],
    "debt_ratio": [
        0.15, 0.20, 0.25, 0.30, 0.35, 0.40,
        0.45, 0.50, 0.55, 0.60, 0.22, 0.38,
        0.48, 0.58, 0.28, 0.42, 0.12, 0.52,
        0.36, 0.65, 0.24, 0.46, 0.57, 0.62
    ],
    "late_payments": [
        0, 0, 0, 0, 1, 1, 1, 2,
        2, 3, 0, 1, 2, 3, 0, 1,
        0, 2, 1, 3, 0, 1, 2, 3
    ],
    "risk": [
        0, 0, 0, 0, 0, 1, 1, 1,
        1, 1, 0, 0, 1, 1, 0, 0,
        0, 1, 0, 1, 0, 1, 1, 1
    ]
})

X = data[[
    "credit_score", "income", "debt_ratio", "late_payments"
]]
y = data["risk"]

credit_model = RandomForestClassifier(
    n_estimators=100,
    random_state=42,
    class_weight="balanced"
)

credit_model.fit(X, y)

print("Credit risk model trained successfully!")
print("Training samples:", len(data))
print("Features:", list(X.columns))


Credit risk model trained successfully!
Training samples: 24
Features: ['credit_score', 'income', 'debt_ratio', 'late_payments']


In [3]:

from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

financial_docs = [
    {
        "id": "F1",
        "source": "Financial Education",
        "text": "A credit score summarizes credit history. Paying bills on time and maintaining responsible credit usage may help improve a credit score."
    },
    {
        "id": "F2",
        "source": "Financial Education",
        "text": "Debt-to-income ratio compares monthly debt payments with gross monthly income. Calculate it as monthly debt payments divided by gross monthly income multiplied by 100."
    },
    {
        "id": "F3",
        "source": "Financial Education",
        "text": "Credit utilization measures the amount of revolving credit used compared with total available credit. Lower utilization is generally preferable."
    },
    {
        "id": "F4",
        "source": "Financial Education",
        "text": "Loan approval depends on lender requirements, income, credit history, existing debts, and repayment capacity. No credit score guarantees loan approval."
    },
    {
        "id": "F5",
        "source": "Financial Education",
        "text": "A budget helps track income and spending. An emergency fund can help manage unexpected expenses."
    },
    {
        "id": "F6",
        "source": "Financial Education",
        "text": "Review your credit report for errors and contact the relevant credit bureau to dispute inaccurate information."
    }
]

doc_texts = [doc["text"] for doc in financial_docs]

rag_vectorizer = TfidfVectorizer(
    stop_words="english",
    ngram_range=(1, 2)
)

rag_matrix = rag_vectorizer.fit_transform(doc_texts)

def retrieve_documents(question, top_k=3):
    q_vector = rag_vectorizer.transform([question])
    scores = cosine_similarity(q_vector, rag_matrix)[0]
    ranked = scores.argsort()[::-1]

    results = []
    for idx in ranked:
        if scores[idx] <= 0:
            continue
        results.append({
            **financial_docs[idx],
            "score": round(float(scores[idx]), 3)
        })
        if len(results) >= top_k:
            break

    return results

print("RAG knowledge base indexed:", len(financial_docs), "documents")


RAG knowledge base indexed: 6 documents


In [7]:

import torch
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM

MODEL_NAME = "google/flan-t5-small"

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
llm_model = AutoModelForSeq2SeqLM.from_pretrained(MODEL_NAME)

device = "cuda" if torch.cuda.is_available() else "cpu"
llm_model = llm_model.to(device)

print("LLM loaded successfully on", device)


Loading weights:   0%|          | 0/190 [00:00<?, ?it/s]

[transformers] The tied weights mapping and config for this model specifies to tie shared.weight to lm_head.weight, but both are present in the checkpoints with different values, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning.


LLM loaded successfully on cpu


In [8]:

def generate_llm_answer(question, credit_info, evidence):
    if not evidence:
        return (
            "I could not find supporting evidence in the current "
            "knowledge base. Please add relevant financial documents "
            "and try again."
        )

    evidence_text = "\n".join(
        f"- {doc['text']} [Evidence: {doc['id']}]"
        for doc in evidence
    )

    prompt = f"""
You are a careful financial education assistant.

User question: {question}

Credit model output:
{credit_info}

Retrieved financial evidence:
{evidence_text}

Instructions:
Answer in simple English.
Use the retrieved evidence to explain the answer.
Do not invent facts, interest rates, or guarantees.
Explain that the credit risk estimate is not a loan approval.
If the evidence does not answer the question, say so.
Mention relevant evidence IDs.
This is educational information, not personalized financial advice.

Answer:
"""

    inputs = tokenizer(
        prompt,
        return_tensors="pt",
        truncation=True,
        max_length=512
    ).to(device)

    with torch.no_grad():
        output_ids = llm_model.generate(
            **inputs,
            max_new_tokens=160,
            do_sample=False
        )

    return tokenizer.decode(
        output_ids[0],
        skip_special_tokens=True
    )


def integrated_financial_assistant(
    question,
    credit_score,
    annual_income,
    debt_ratio_percent,
    late_payments
):
    if not question or not question.strip():
        return "Enter a financial question.", "", ""


    if not 300 <= credit_score <= 850:
        return "Credit score must be between 300 and 850.", "", ""

    if annual_income <= 0:
        return "Annual income must be greater than zero.", "", ""

    if not 0 <= debt_ratio_percent <= 100:
        return "Debt ratio must be between 0 and 100.", "", ""

    if not 0 <= late_payments <= 20:
        return "Late payments must be between 0 and 20.", "", ""


    profile = pd.DataFrame([{
        "credit_score": credit_score,
        "income": annual_income,
        "debt_ratio": debt_ratio_percent / 100,
        "late_payments": late_payments
    }])

    risk_prediction = int(credit_model.predict(profile)[0])
    risk_probability = float(
        credit_model.predict_proba(profile)[0][1]
    )

    risk_label = (
        "Higher predicted risk"
        if risk_prediction == 1
        else "Lower predicted risk"
    )

    credit_info = (
        f"Risk category: {risk_label}. "
        f"Model risk score: {risk_probability:.1%}. "
        "This is a toy model trained on synthetic sample data, "
        "not a validated probability of default."
    )


    evidence = retrieve_documents(question, top_k=3)

    sources = "\n".join(
        f"- {doc['id']}: {doc['source']} — {doc['text']}"
        for doc in evidence
    ) if evidence else "No relevant evidence found."


    answer = generate_llm_answer(
        question, credit_info, evidence
    )


    credit_output = (
        f"**Predicted category:** {risk_label}\n\n"
        f"**Model risk score:** {risk_probability:.1%}\n\n"
        "This is a demonstration prediction, not a lending decision."
    )

    return answer, credit_output, sources

print("Credit Model + RAG + LLM integration ready!")


Credit Model + RAG + LLM integration ready!


In [ ]:

import gradio as gr

with gr.Blocks(theme=gr.themes.Soft()) as demo:
    gr.Markdown(
        """
        # 💰 Integrated Financial RAG Assistant

        Credit Risk Model + Evidence Retrieval + LLM

        Enter a sample credit profile and ask a financial question.
        """
    )

    with gr.Row():
        credit_score = gr.Slider(
            300, 850, value=680, step=1,
            label="Credit Score"
        )
        annual_income = gr.Number(
            value=50000, label="Annual Income"
        )

    with gr.Row():
        debt_ratio = gr.Slider(
            0, 100, value=30, step=1,
            label="Debt-to-Income Ratio (%)"
        )
        late_payments = gr.Slider(
            0, 10, value=0, step=1,
            label="Late Payments"
        )

    question = gr.Textbox(
        label="Financial Question",
        placeholder="Example: How can I improve my credit score?",
        lines=2
    )

    submit = gr.Button("Analyze and Answer", variant="primary")

    answer_output = gr.Markdown(label="LLM Explanation")
    credit_output = gr.Markdown(label="Credit Model Result")
    sources_output = gr.Markdown(label="Retrieved Evidence")

    submit.click(
        fn=integrated_financial_assistant,
        inputs=[
            question,
            credit_score,
            annual_income,
            debt_ratio,
            late_payments
        ],
        outputs=[
            answer_output,
            credit_output,
            sources_output
        ]
    )

demo.launch(share=True, debug=True)


/tmp/ipykernel_4720/1868838807.py:3: UserWarning: The parameters have been moved from the Blocks constructor to the launch() method in Gradio 6.0: theme. Please pass these parameters to launch() instead.
  with gr.Blocks(theme=gr.themes.Soft()) as demo:


Colab notebook detected. This cell will run indefinitely so that you can see errors and logs. To turn off, set debug=False in launch().
* Running on public URL: https://abf7f043fc9cc40f72.gradio.live

This share link is temporary and will last for up to 1 week (best effort). For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)
